# 6  Model validation: finding a leaking input

**Goal:** recognise a result that is too good to be true, find the input that leaks the answer, and validate the model with cross-validation.

**Data:** 6,000 synthetic residency and visa applications. Target: will the application be late? The prediction is made **when the application is submitted**.

**The data used here:**
- `late_applications.csv`: [view as a table](https://github.com/u1i/ml-foundations-workshop/blob/main/data/late_applications.csv) (opens on GitHub) · [download](https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/late_applications.csv) (right-click, *Save link as*)

**How this works:** each step is one cell. Click the &#9654; button on the left of the cell and read what appears below it. Where a cell has a field on the right, choose an option and click &#9654; again. No coding is needed. The cells marked *Stretch* at the end are optional and for those who like to code.

All data is synthetic.

In [ ]:
#@title Setup  (Click &#9654; first)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
DATA = "https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
RED, INK, GREY, GREEN, BLUE = "#A12527", "#1d1d1b", "#6e6e6a", "#2e7d4f", "#1f5a8a"
_STYLE = {"info": ("#1f5a8a", "#eef4f9"), "see": ("#A12527", "#fbeeee"), "ok": ("#2e7d4f", "#e9f5ee"),
          "try": ("#9a6a00", "#fbf3dd"), "step": ("#1d1d1b", "#f6f5f2")}

def box(title, text="", kind="info"):
    c, bg = _STYLE[kind]
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;border-left:6px solid {c};background:{bg};'
                 f'color:#1d1d1b;padding:14px 20px;margin:12px 0;font-size:17px;line-height:1.5;max-width:900px">'
                 f'<div style="font-size:20px;font-weight:bold;color:{c};margin-bottom:4px">{title}</div>{text}</div>'))

def big(*items):
    """big((value, label), (value, label, colour), ...): large numbers in one row."""
    cells = "".join(f'<div style="display:inline-block;margin:6px 40px 6px 0;vertical-align:top"><div style="font-size:15px;color:#6e6e6a">{it[1]}</div>'
                    f'<div style="font-size:44px;font-weight:bold;color:{it[2] if len(it) > 2 else INK}">{it[0]}</div></div>' for it in items)
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;color:#1d1d1b">{cells}</div>'))

def answer(choice, options):
    """options: {option: (is_right, explanation)}; shows feedback for the chosen option."""
    if choice not in options:
        box("Make your choice", "Pick an option in the field on the right of this cell, then click &#9654; again.", "try")
        return False
    right, why = options[choice]
    box("Correct" if right else "Not quite", why, "ok" if right else "try")
    return right

def table(df, highlight=None, n=10):
    """Show rows with cells highlighted: highlight = {(row_label, column): colour}."""
    sub = df.head(n).astype(object).where(df.head(n).notna(), "(empty)")
    def colour(_):
        out = pd.DataFrame("", index=sub.index, columns=sub.columns)
        for (r, c), col in (highlight or {}).items():
            if r in out.index and c in out.columns:
                out.loc[r, c] = f"background-color:{col};font-weight:bold"
        return out
    display(sub.style.apply(colour, axis=None).set_table_styles([{"selector": "td,th", "props": "font-size:14px;padding:4px 10px"}]))
import ipywidgets as W
def fslider(desc, lo, hi, step, val, fmt=".2f"):
    return W.FloatSlider(value=val, min=lo, max=hi, step=step, description=desc, continuous_update=False, readout_format=fmt,
                         layout=W.Layout(width="700px"), style={"description_width": "200px"})
def islider(desc, lo, hi, val):
    return W.IntSlider(value=val, min=lo, max=hi, description=desc, continuous_update=False,
                       layout=W.Layout(width="700px"), style={"description_width": "200px"})
live = W.interact

def matrix(y_true, y_flag, labels=("honest", "fraud"), flag_word="flagged"):
    """Confusion matrix as four large boxes."""
    y_true, y_flag = np.asarray(y_true).astype(bool), np.asarray(y_flag).astype(bool)
    tp, fn = int((y_true & y_flag).sum()), int((y_true & ~y_flag).sum())
    fp, tn = int((~y_true & y_flag).sum()), int((~y_true & ~y_flag).sum())
    cell = lambda n, t, c, bg: (f'<td style="border:2px solid #1d1d1b;padding:14px 18px;background:{bg};width:260px">'
                                f'<div style="font-size:15px;color:{c};font-weight:bold">{t}</div><div style="font-size:38px;font-weight:bold;color:{c}">{n:,}</div></td>')
    display(HTML(f'<table style="font-family:Arial,Helvetica,sans-serif;border-collapse:collapse;color:#1d1d1b;margin:8px 0">'
                 f'<tr><td></td><th style="padding:6px;font-size:15px">Model: {flag_word}</th><th style="padding:6px;font-size:15px">Model: not {flag_word}</th></tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[1]}</th>{cell(tp, "Found", GREEN, "#e9f5ee")}{cell(fn, "Missed", RED, "#fbeeee")}</tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[0]}</th>{cell(fp, "Checked for nothing", RED, "#fbeeee")}{cell(tn, "Correctly left alone", GREEN, "#e9f5ee")}</tr></table>'))
    return tp, fn, fp, tn

def bars(values, title, highlight=None, fmt="{:.2f}", color=INK):
    """Horizontal bars with values; values is a dict or Series."""
    s = pd.Series(values)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1))
    ax.barh(s.index.astype(str), s.values, color=[RED if k == highlight else color for k in s.index])
    for i, v in enumerate(s.values): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=12)
    ax.invert_yaxis(); ax.set_title(title); plt.show()


box("Ready", "Setup done. Continue with Step 1 below.", "ok")

## Step 1  Too good to be true

In [ ]:
#@title Step 1  A colleague's model
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
df = pd.read_csv(DATA + "late_applications.csv", parse_dates=["documents_uploaded_at", "submitted_at", "final_approval_date"])
df["days_until_final_approval"] = (df["final_approval_date"] - df["submitted_at"].dt.normalize()).dt.days
df["submitted_month"] = df["submitted_at"].dt.month
INPUTS = ["service", "channel", "number_of_documents", "documents_complete", "medical_fitness_document_required", "additional_document_review_required", "queue_at_submission", "submitted_month", "days_until_final_approval"]
X = pd.get_dummies(df[INPUTS], dtype=int); y = df["late"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
m1 = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
acc1 = m1.score(X_test, y_test)
big((f"{acc1:.1%}", "right on the test data", RED))
box("Too good to be true?", "Real services are messy. A model that is right almost every time about delays is a reason to check, not to celebrate.", "see")

In [ ]:
#@title Step 2  What does the model rely on?
imp = pd.Series(m1.feature_importances_, index=X.columns).sort_values(ascending=False).head(8)
bars(imp, "Feature importance", highlight=imp.index[0], fmt="{:.0%}")
box("Do you see it?", f"One input carries almost everything: <b>{imp.index[0]}</b>. The model has found a shortcut.", "see")

## Step 3  Find the leaking input

In [ ]:
#@title Step 3  Which input is only known after the decision?  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "queue_at_submission", "additional_document_review_required", "days_until_final_approval", "submitted_month"]
_ = answer(question, {
    "queue_at_submission": (False, "The queue is counted when the application arrives, so it is known in time."),
    "additional_document_review_required": (False, "Whether an additional document review is needed is set by the rules when the application is submitted."),
    "days_until_final_approval": (True, "It is computed from the final approval date, which only exists once the application is decided. When the prediction is needed, it is not there yet: it leaks the answer."),
    "submitted_month": (False, "The month of submission is known on the day the application arrives."),
})

In [ ]:
#@title Step 4  Remove it and train again
CLEAN = [c for c in INPUTS if c != "days_until_final_approval"]
Xc = pd.get_dummies(df[CLEAN], dtype=int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, y, test_size=0.2, random_state=42)
m2 = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xc_train, yc_train)
acc2 = m2.score(Xc_test, yc_test)
big((f"{acc1:.1%}", "with the leaking input", RED), (f"{acc2:.1%}", "without it", GREEN))
box("The honest number", f"{acc2:.0%} is lower, and it is what the model will really do in the service. A model with the leaking input would fail on its first day, because the approval date does not exist yet.", "see")

## Step 5  Cross-validation

In [ ]:
#@title Step 5  Five rounds instead of one split
scores = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42), Xc_train, yc_train, cv=5)
fig, ax = plt.subplots(figsize=(10, 3.2))
for r in range(5):
    for p in range(5): ax.add_patch(plt.Rectangle((p, 4 - r), .95, .8, color=RED if p == r else "#dddddd"))
    ax.text(5.2, 4 - r + .4, f"round {r + 1}: {scores[r]:.3f}", va="center", fontsize=13)
ax.set_xlim(0, 7); ax.set_ylim(-.2, 5); ax.axis("off"); ax.set_title("Each part is the test part once (red)"); plt.show()
big((f"{scores.mean():.3f}", "average"), (f"±{scores.std():.3f}", "spread"))
box("What it tells you", "The average shows how good the model is; the spread shows how stable it is. One lucky split cannot fool you any more.", "info")

## Step 6  Tune the depth with cross-validation

In [ ]:
#@title Step 6  Move the slider  (then move the slider)
D = list(range(1, 21))
trn = [DecisionTreeClassifier(max_depth=d, random_state=42).fit(Xc_train, yc_train).score(Xc_train, yc_train) for d in D]
cv = [cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=42), Xc_train, yc_train, cv=5).mean() for d in D]
best = D[int(np.argmax(cv))]
def show(depth):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(D, [1 - a for a in trn], color=INK, lw=3, label="error on training data"); ax.plot(D, [1 - c for c in cv], color=RED, lw=3, label="cross-validation error")
    ax.axvline(best, color=GREEN, ls="--", lw=2, label=f"best depth: {best}"); ax.scatter([depth], [1 - cv[depth - 1]], s=140, color=RED, zorder=3)
    ax.set_xlabel("depth of the tree"); ax.set_ylabel("error"); ax.legend(); plt.show()
    big((depth, "depth"), (f"{1 - trn[depth - 1]:.3f}", "training error"), (f"{1 - cv[depth - 1]:.3f}", "cross-validation error", RED))
_ = live(show, depth=islider("Depth of the tree", 1, 20, 3))

## Result

In [ ]:
#@title Result
box("Do you see it now?", "A model that looked perfect was using an input from the future. After removing it, the model is honest, "
    "and cross-validation shows how good and how stable it is. <b>Now the service can trust the number it reports.</b> "
    "The question to ask of every input: is it known at the moment the prediction is made?", "see")

---
## Stretch (optional, for those who code)

In [ ]:
# Stretch: compare the spread of the cross-validation results with and without the leaking input
with_leak = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42), X_train, y_train, cv=5)
print("with the leaking input:   ", with_leak.round(3), "spread", with_leak.std().round(3))
print("without the leaking input:", scores.round(3), "spread", scores.std().round(3))